# Imports

In [1]:
import numpy as np
import pandas as pd
from pycaret.anomaly import AnomalyExperiment
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import sqlite3
from pyod.models.abod import ABOD
from pyod.models.cblof import CBLOF
from sklearn.svm import OneClassSVM
from pyod.models.iforest import IForest
from pyod.models.hbos import HBOS
from itertools import product
from datetime import timedelta
from time import perf_counter
from tqdm.auto import tqdm

# Global Constants

Random seed to reproduce randomness.

In [2]:
RANDOM_SEED = 42

# Data

## Load Data

Connect to sqlite database.

In [3]:
conn = sqlite3.connect("../data/data.db")
cursor = conn.cursor()

Load the dataset from the database (random without SEED).

In [4]:
# Variable used to collect data from the database (BE CAREFUL NOT TO EXCEED MEMORY)
data_quantity = 1000000

In [5]:
# Collects data from the bank based on the label and limiting the quantity to avoid problems with full memory.
df = pd.read_sql_query("SELECT * FROM (SELECT * FROM packets_flow_features WHERE label = 'Benign' ORDER BY RANDOM() LIMIT ?) UNION ALL SELECT * FROM (SELECT * FROM packets_flow_features WHERE label != 'Benign' ORDER BY RANDOM() LIMIT ?);", conn, params=(data_quantity*2, data_quantity/2))

df

,destination_port,flow_duration,total_foward_packets,total_backward_packets,total_length_of_foward_packets,total_length_of_backward_packets,foward_packet_length_max,foward_packet_length_min,foward_packet_length_average,foward_packet_length_std,...,min_segment_size_forward,active_average,active_std,active_max,active_min,idle_average,idle_std,idle_max,idle_min,label
0,54072,4193513,None,None,None,None,935,0,187.0,418.1447118,...,None,0,0,0,0,0,0,0,0,Benign
1,2179,69,2,2,4,12.0,2,2,2.0,0.0,...,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,52292,7,None,None,None,None,0.0,0.0,0.0,0.0,...,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,2449,86405426,None,None,None,None,0,0,0.0,0.0,...,None,0.0,0.0,0,0,86405426.0,0.0,86405426,86405426,Benign
4,445,40075298,None,None,None,None,140.0,0.0,74.6,70.2837107728384,...,None,10310209.0,0.0,10310209.0,10310209.0,14882544.5,7015737.41334441,19843420.0,9921669.0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499995,52017,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1062718975,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DDoS SNMP
2499996,45900,30897501,None,5,0.0,0.0,0.0,0.0,0.0,0.0,...,None,0.0,0.0,0.0,0.0,7571578.0,2141174.65411395,10655689.0,5801011.0,DoS SYN Flood
2499997,26667,357783,None,1,0.0,0.0,0.0,0.0,0.0,0.0,...,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DoS SYN Flood
2499998,57368,5999697,6,0,3096.0,0.0,516.0,516.0,516.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TFTP Attack


In [6]:
df.columns

Index(['destination_port', 'flow_duration', 'total_foward_packets',
       'total_backward_packets', 'total_length_of_foward_packets',
       'total_length_of_backward_packets', 'foward_packet_length_max',
       'foward_packet_length_min', 'foward_packet_length_average',
       'foward_packet_length_std', 'backward_packet_length_max',
       'backward_packet_length_min', 'backward_packet_length_average',
       'backward_packet_length_std', 'flow_bytes/s', 'flow_packets/s',
       'flow_iat_average', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'foward_iat_total', 'foward_iat_average', 'foward_iat_std',
       'foward_iat_max', 'foward_iat_min', 'backward_iat_total',
       'backward_iat_average', 'backward_iat_std', 'backward_iat_max',
       'backward_iat_min', 'foward_psh_flags', 'backward_psh_flags',
       'foward_urg_flags', 'backward_urg_flags', 'foward_header_length',
       'backward_header_length', 'foward_packets/s', 'backward_packets/s',
       'min_packet_length

In [7]:
df["label"].unique()

array(['Benign', 'DoS SYN Flood', 'TFTP Attack', 'DoS ACK Fragmentation',
       'DoS UDP Flood', 'MSSQL', 'DDoS SNMP', 'DDoS RSTFIN Flood',
       'DoS TCP Flood', 'DDoS DNS Flood', 'DoS HTTP Flood',
       'DDoS UDP Flood', 'Brute Force SSH', 'DDoS MSSQL',
       'DDoS HTTP Flood', 'DDoS SSDP', 'DDoS', 'DDoS NTP', 'Infiltration',
       'DDoS NetBIOS', 'Brute Force Dictionary',
       'Reconnaissance Port Scanning', 'DDoS ACK Fragmentation', 'Bot',
       'NetBIOS', 'DoS MQTT Connect Flood', 'DoS ICMP Flood', 'LDAP',
       'DDoS MQTT Publish Flood', 'DoS Hulk', 'DDoS LDAP', 'Scanning',
       'DDoS LOIC HTTP', 'Backdoor', 'DDoS SYN Flood', 'DoS ACK Flood',
       'DDoS PSHACK Flood', 'DDoS HOIC', 'DoS ICMP Fragmentation',
       'Brute Force FTP', 'UDP Lag', 'Web Attack XSS', 'Brute Force MQTT',
       'MITM', 'Unknown', 'Reconnaissance OS Fingerprinting',
       'Password Attack', 'DoS Mirai Flood', 'Reconnaissance',
       'DoS SlowHTTPTest', 'MITM DNS Spoofing', 'DoS Slowloris',


## Data Clean and Reload

Perform some pre-training data cleaning. The preprocessing done previously is not enough to make the data ready for training.

### Column Drop

Drop the destination_port column, as the difference in door numbers has nothing to do with the difference between the doors

In [8]:
df.drop("destination_port", axis=1, inplace=True)

Check if columns have a standard deviation equal to zero, which causes errors in the data normalization process.

In [9]:
def check_std_dev(df):
    # Check standard deviation of numeric columns
    std_dev = df.select_dtypes(include="number").std()

    # Show columns with zero standard deviation
    zero_std = std_dev[std_dev == 0]
    print("Columns with zero standard deviation:")
    print(zero_std)

In [10]:
check_std_dev(df)

Columns with zero standard deviation:
Series([], dtype: float64)


### Duplicate Removal

Remove duplicate data, as they are redundant and will only make training more expensive, without resulting in relevant learning for the model.

In [11]:
def df_duplicate_removal(df):
    initial_len = df.shape[0]
    df = df.drop_duplicates()
    print(f"Initial size: {initial_len}, final size {df.shape[0]} | Discarded {initial_len - df.shape[0]} duplicates")

    df = df.reset_index(drop=True) # Reset index
    return df

In [12]:
df = df_duplicate_removal(df)

Initial size: 2500000, final size 1832002 | Discarded 667998 duplicates


### Handle Invalid Values

Drop lines with infinite and NaN from the dataframe.

In [13]:
# Function to remove lines with infinite and NaN values from a DataFrame
def df_remove_inf_nan(df, reset_index=True):
    # Transform infinite values to NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    initial_length = len(df)

    nan_rows = df.isna().any(axis=1)
    removed_indices = df[nan_rows].index.tolist()

    # Drop lines with NaN values
    df.dropna(inplace=True)

    print(f"Removed {initial_length - len(df)} rows with NaN or infinite values")

    # Reset index
    if reset_index:
        df.reset_index(drop=True, inplace=True)
    else:
        return df, removed_indices

    return df

In [14]:
# Remove lines with infinite and NaN values from the dataframe
df = df_remove_inf_nan(df)

Removed 1379879 rows with NaN or infinite values


### Load More Data

A lot of data was removed during cleaning, so now there is more data in the dataframe, the previous steps will be redone to have more data in the dataframe up to a data limit (so as not to fill the memory).

In [15]:
while True:
    # Repeat the previous query
    chunk = pd.read_sql_query("SELECT * FROM (SELECT * FROM packets_flow_features WHERE label = 'Benign' ORDER BY RANDOM() LIMIT ?) UNION ALL SELECT * FROM (SELECT * FROM packets_flow_features WHERE label != 'Benign' ORDER BY RANDOM() LIMIT ?);", conn, params=(data_quantity, data_quantity/4))
    # Concatenates the main dataframe with the new chunk
    df = pd.concat([df, chunk])
    # Remove duplicates
    df = df_duplicate_removal(df)
    # Remove invalid values
    df = df_remove_inf_nan(df)

    print(f"Current size: {len(df)}")
    # Limit
    if len(df) >= data_quantity:
        del chunk
        break

Initial size: 1702123, final size 1572987 | Discarded 129136 duplicates
Removed 1241585 rows with NaN or infinite values
Current size: 331402
Initial size: 1581402, final size 1440133 | Discarded 141269 duplicates
Removed 788586 rows with NaN or infinite values
Current size: 651547
Initial size: 1901547, final size 1748852 | Discarded 152695 duplicates
Removed 789000 rows with NaN or infinite values
Current size: 959852
Initial size: 2209852, final size 2048042 | Discarded 161810 duplicates
Removed 789547 rows with NaN or infinite values
Current size: 1258495


### Transform String to Numeric

Transform numbers that are in string to some numeric data type.

In [16]:
def df_transform_string_to_numeric(df, text_columns=None, flag_remove_inf_nan=False):

    if text_columns:
        # Separate columns to convert
        numeric_cols = df.columns.difference(text_columns)

        # Apply conversion to numeric columns
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    else:
        df = df.apply(pd.to_numeric, errors='coerce')

    if flag_remove_inf_nan:
        # Remove lines with infinite and NaN values from the dataframe
        df = df_remove_inf_nan(df)

    return df

In [17]:
# Columns to ignore (keep as text)
text_columns = ['label']

In [18]:
df = df_transform_string_to_numeric(df, text_columns, True)

Removed 16126 rows with NaN or infinite values


### Remove Correlated Features

Discard features with high correlation to avoid to pass redundant information to the model. This way, we will be able to obtain a simpler model with lower computational cost.

In [19]:
def get_highly_correlated_features(correlation_matrix, threshold=0.95):
  correlated_pairs = []
  for i in range(len(correlation_matrix.columns)):
    for j in range(i):
      if abs(correlation_matrix.iloc[i, j]) > threshold:
        pair = (correlation_matrix.columns[i], correlation_matrix.columns[j])
        coefficient = correlation_matrix.iloc[i, j]
        correlated_pairs.append((pair, coefficient))
  return sorted(correlated_pairs, key= lambda pair: pair[1], reverse=True)

In [20]:
def df_remove_high_correlated_features(df, threshold=0.95):
    corr_matrix = df.select_dtypes(include=['number']).corr().abs()
    correlation_list = get_highly_correlated_features(corr_matrix, threshold)
    print(correlation_list)

    f2drop = []
    for feature_pair, _ in correlation_list:
        if feature_pair[0] not in f2drop and feature_pair[1] not in f2drop:
            f2drop.append(feature_pair[1])

    df = df.drop(f2drop, axis='columns')

    return df

In [21]:
#df = df_remove_high_correlated_features(df)

## Data Organization

Organization of data to leave it in the model input format. In addition, the data is separated into training, validation and test sets..

### Filter Data

Filter data based on benign/malicious.

In [22]:
# Drop the "label" column as it is not needed for numeric conversion and convert all columns to numeric, coercing errors to NaN (non-numeric values become NaN)
X_benign = df[df["label"] == "Benign"].drop(columns=["label"], axis=1)
X_malicious = df[df["label"] != "Benign"].drop(columns=["label"], axis=1)

In [23]:
del df

In [24]:
print(f"Benigns: {len(X_benign)}", f"Malicious: {len(X_malicious)}")

Benigns: 692449 Malicious: 549920


### Split: Train/Validation/Test

Separation of the dataset into training, validation and test.

In [25]:
# Split: train_val/test
X_benign_train_val, X_benign_test = train_test_split(
    X_benign,
    test_size=0.1,
    random_state=RANDOM_SEED
    )

In [26]:
del X_benign

In [27]:
# Split: train/val
X_train, X_benign_val = train_test_split(
    X_benign_train_val,
    test_size=0.1,
    random_state=RANDOM_SEED
    )

In [28]:
del X_benign_train_val

Mixing benign and malignant in val and test, to make a more complete training.

In [29]:
split_idx = int(len(X_malicious) * 0.9)

X_malicious_1 = X_malicious.iloc[:split_idx]
X_malicious_2 = X_malicious.iloc[split_idx:]

In [30]:
X_val = pd.concat([X_benign_val, X_malicious_2], ignore_index=True)
y_val = np.concatenate([np.zeros(len(X_benign_val)), np.ones(len(X_malicious_2))])

In [31]:
X_test = pd.concat([X_benign_test, X_malicious_1], ignore_index=True)
y_test = np.concatenate([np.zeros(len(X_benign_test)), np.ones(len(X_malicious_1))])

In [32]:
del X_benign_test
del X_benign_val
del X_malicious
del split_idx

### Normalization

Apply normalization.

In [33]:
# Scaler initialization
scaler = StandardScaler()

In [34]:
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# AI Models

Artificial intelligence models.

### General Functions

In [ ]:
def print_classification_report(y_true, y_pred, threshold=None):
    if threshold is None:
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))

        print("Accuracy:", accuracy_score(y_true, y_pred))
        print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

## AutoML - Pycaret

Using the Pycaret framework to find out which algorithms perform best, and then create manual models of these algorithms.

In [ ]:
anomaly_exp = AnomalyExperiment()

anomaly_exp.setup(
    data=X_train,
    normalize=False,
    preprocess=True,
    session_id=RANDOM_SEED,
    use_gpu=True,
    verbose=True
)

In [ ]:
anomaly_exp.models()

Train and test each model.

In [ ]:
for model in anomaly_exp.models().index:
    try:
        print("--------------------------------")
        print(f"\nModel ID: {model}")
        
        model = anomaly_exp.create_model(model)
        
        print(f"Model: {model}")

        predictions = anomaly_exp.predict_model(model, data=X_test)
        y_pred = predictions['Anomaly']

        # Evaluation metrics (like precision, recall, F1-score, etc.)
        print_classification_report(y_test, y_pred)
    except Exception as e:
        print(f"ERROR: {e}")

**Pycaret Models Test Output:**

Model ID: abod
Model: ABOD(contamination=0.05, method='fast', n_neighbors=5)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.59      0.95      0.73     60813
         1.0       0.99      0.92      0.95    482853

    accuracy                           0.92    543666
   macro avg       0.79      0.93      0.84    543666
weighted avg       0.95      0.92      0.93    543666

Accuracy: 0.9193273075748714
Confusion Matrix:
 [[ 57900   2913]
 [ 40946 441907]]
--------------------------------

Model ID: cluster
Model: CBLOFForceToDouble(alpha=0.9, beta=5, check_estimator=False,
          clustering_estimator=None, contamination=0.05, n_clusters=8,
          n_jobs=None, random_state=42, use_weights=False)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.51      0.95      0.66     60813
         1.0       0.99      0.88      0.93    482853

    accuracy                           0.89    543666
   macro avg       0.75      0.92      0.80    543666
weighted avg       0.94      0.89      0.90    543666

Accuracy: 0.8906497739420894
Confusion Matrix:
 [[ 57783   3030]
 [ 56420 426433]]
--------------------------------

Model ID: cof
Initiated	. . . . . . . . . . . . . . . . . .	02:04:39
Status	. . . . . . . . . . . . . . . . . .	Fitting 0.05 Fraction
Estimator	. . . . . . . . . . . . . . . . . .	Connectivity-Based Local Outlier
ERROR: Unable to allocate 139. GiB for an array with shape (136829, 136829) and data type float64
--------------------------------

Model ID: iforest
Model: IForest(behaviour='new', bootstrap=False, contamination=0.05,
    max_features=1.0, max_samples='auto', n_estimators=100, n_jobs=-1,
    random_state=42, verbose=0)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.12      0.95      0.21     60813
         1.0       0.95      0.13      0.23    482853

    accuracy                           0.22    543666
   macro avg       0.54      0.54      0.22    543666
weighted avg       0.86      0.22      0.22    543666

Accuracy: 0.22004870637486987
Confusion Matrix:
 [[ 57785   3028]
 [421005  61848]]
--------------------------------

Model ID: histogram
Model: HBOS(alpha=0.1, contamination=0.05, n_bins=10, tol=0.5)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.11      0.95      0.19     60813
         1.0       0.50      0.01      0.01    482853

    accuracy                           0.11    543666
   macro avg       0.30      0.48      0.10    543666
weighted avg       0.46      0.11      0.03    543666

Accuracy: 0.11185176192735981
Confusion Matrix:
 [[ 57832   2981]
 [479875   2978]]
--------------------------------

Model ID: knn
Model: KNN(algorithm='auto', contamination=0.05, leaf_size=30, method='largest',
  metric='minkowski', metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
  radius=1.0)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.58      0.95      0.72     60813
         1.0       0.99      0.91      0.95    482853

    accuracy                           0.92    543666
   macro avg       0.78      0.93      0.83    543666
weighted avg       0.95      0.92      0.92    543666

Accuracy: 0.9163935210220981
Confusion Matrix:
 [[ 57820   2993]
 [ 42461 440392]]
--------------------------------

Model ID: lof
Model: LOF(algorithm='auto', contamination=0.05, leaf_size=30, metric='minkowski',
  metric_params=None, n_jobs=-1, n_neighbors=20, novelty=True, p=2)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.42      0.95      0.58     60813
         1.0       0.99      0.83      0.91    482853

    accuracy                           0.85    543666
   macro avg       0.70      0.89      0.74    543666
weighted avg       0.93      0.85      0.87    543666

Accuracy: 0.8457472050854753
Confusion Matrix:
 [[ 57815   2998]
 [ 80864 401989]]
--------------------------------

Model ID: svm
Model: OCSVM(cache_size=200, coef0=0.0, contamination=0.05, degree=3, gamma='auto',
   kernel='rbf', max_iter=-1, nu=0.5, shrinking=True, tol=0.001,
   verbose=False)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.45      0.95      0.61     60813
         1.0       0.99      0.86      0.92    482853

    accuracy                           0.87    543666
   macro avg       0.72      0.90      0.77    543666
weighted avg       0.93      0.87      0.89    543666

Accuracy: 0.8666423870538161
Confusion Matrix:
 [[ 57812   3001]
 [ 69501 413352]]
--------------------------------

Model ID: pca
Model: PCA(contamination=0.05, copy=True, iterated_power='auto', n_components=None,
  n_selected_components=None, random_state=42, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.38      0.95      0.54     60813
         1.0       0.99      0.80      0.89    482853

    accuracy                           0.82    543666
   macro avg       0.68      0.88      0.71    543666
weighted avg       0.92      0.82      0.85    543666

Accuracy: 0.817373166613325
Confusion Matrix:
 [[ 57777   3036]
 [ 96252 386601]]
--------------------------------

Model ID: mcd
Model: MCD(assume_centered=False, contamination=0.05, random_state=42,
  store_precision=True, support_fraction=None)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.11      0.95      0.20     60813
         1.0       0.88      0.05      0.09    482853

    accuracy                           0.15    543666
   macro avg       0.50      0.50      0.14    543666
weighted avg       0.79      0.15      0.10    543666

Accuracy: 0.14670404255553962
Confusion Matrix:
 [[ 57814   2999]
 [460909  21944]]
--------------------------------

Model ID: sod
Model: SOD(alpha=0.8, contamination=0.05, n_neighbors=20, ref_set=10)
ERROR: 0
--------------------------------

Model ID: sos
Initiated	. . . . . . . . . . . . . . . . . .	09:55:33
Status	. . . . . . . . . . . . . . . . . .	Fitting 0.05 Fraction
Estimator	. . . . . . . . . . . . . . . . . .	Stochastic Outlier Selection
ERROR: Unable to allocate 69.7 GiB for an array with shape (136829, 136829) and data type float32

## 1 - Angle-based Outlier Detector (ABOD)

### Training

In [30]:
model = ABOD(
    contamination=0.05,
    method='fast',
    n_neighbors=5
    )

In [ ]:
model.fit(X_train)

### Validation

In [ ]:
y_val_pred = model.predict(X_val)  # 0 = normal, 1 = anomaly

In [ ]:
print_classification_report(y_val, y_val_pred)

### Test

In [ ]:
y_test_pred = model.predict(X_test)  # 0 = normal, 1 = anomaly

In [ ]:
print_classification_report(y_test, y_test_pred)

## 2 - Cluster-Based Local Outlier Factor (CBLOF)

### Training

In [40]:
model = CBLOF(
    n_clusters=8,
    contamination=0.05,
    alpha=0.5,
    use_weights=False,
    clustering_estimator=None,  # KMeans
    check_estimator=True,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

In [41]:
model.fit(X_train)

CBLOF(alpha=0.5, beta=5, check_estimator=True, clustering_estimator=None,
   contamination=0.1, n_clusters=8, n_jobs=None, random_state=42,
   use_weights=False)

### Validation

In [191]:
y_val_pred = model.predict(X_val)  # 0 = normal, 1 = anomaly

In [192]:
print_classification_report(y_val, y_val_pred)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.90      0.92     62321
         1.0       0.89      0.94      0.92     54992

    accuracy                           0.92    117313
   macro avg       0.92      0.92      0.92    117313
weighted avg       0.92      0.92      0.92    117313

Accuracy: 0.9195400339263339
Confusion Matrix:
 [[56075  6246]
 [ 3193 51799]]


#### Hyperparameter Tuning

Testing of various combinations of various parameters.

In [51]:
n_cluster_list = list(range(2, 50))
#contamination_list = list(np.arange(0.01, 0.20, 0.01))
alpha_list = list(np.arange(0.1, 0.9, 0.1))
use_weights_list = [True, False]
check_estimator_list = [True, False]

In [52]:
param_grid = list(
    product(
        n_cluster_list,
        contamination_list,
        alpha_list,
        use_weights_list,
        check_estimator_list
    )
)

best_accuracy = best_precision = best_recall = best_f1 = 0
best_params = {}

start = perf_counter()  # marca o início

for n_cluster, contamination, alpha, use_weights, check_estimator in tqdm(
    param_grid,
    desc="Grid Search CBLOF",
    unit="comb",
    colour="#2ca02c"         # opcional: cor da barra
):
    model = CBLOF(
        n_clusters=n_cluster,
        contamination=contamination,
        alpha=alpha,
        use_weights=use_weights,
        check_estimator=check_estimator,
        clustering_estimator=None,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    model.fit(X_train)
    y_val_pred = model.predict(X_val)  # 0 = normal, 1 = anomaly

    accuracy  = accuracy_score (y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred)
    recall    = recall_score   (y_val, y_val_pred)
    f1        = f1_score       (y_val, y_val_pred)

    if accuracy > best_accuracy:
        best_accuracy  = accuracy
        best_precision = precision
        best_recall    = recall
        best_f1        = f1
        best_params = {
            "n_cluster": n_cluster,
            "contamination": contamination,
            "alpha": alpha,
            "use_weights": use_weights,
            "check_estimator": check_estimator
        }

        print(f"Melhor acurácia : {best_accuracy :.4f}")
        print(f"Melhor precisão: {best_precision:.4f}")
        print(f"Melhor recall  : {best_recall   :.4f}")
        print(f"Melhor F1      : {best_f1       :.4f}")
        print(f"Melhores parâmetros: {best_params}")

elapsed = timedelta(seconds=perf_counter() - start)  # tempo total

print(f"Melhor acurácia : {best_accuracy :.4f}")
print(f"Melhor precisão: {best_precision:.4f}")
print(f"Melhor recall  : {best_recall   :.4f}")
print(f"Melhor F1      : {best_f1       :.4f}")
print(f"Melhores parâmetros: {best_params}")
print(f"Tempo total: {elapsed}")

Grid Search CBLOF:   0%|          | 0/29184 [00:00<?, ?comb/s]

Melhor acurácia : 0.8877
Melhor precisão: 0.9865
Melhor recall  : 0.7709
Melhor F1      : 0.8655
Melhores parâmetros: {'n_cluster': 2, 'contamination': 0.01, 'alpha': 0.1, 'use_weights': True, 'check_estimator': True}
Melhor acurácia : 0.8939
Melhor precisão: 0.9726
Melhor recall  : 0.7961
Melhor F1      : 0.8755
Melhores parâmetros: {'n_cluster': 2, 'contamination': 0.02, 'alpha': 0.1, 'use_weights': True, 'check_estimator': True}
Melhor acurácia : 0.9138
Melhor precisão: 0.9623
Melhor recall  : 0.8493
Melhor F1      : 0.9023
Melhores parâmetros: {'n_cluster': 2, 'contamination': 0.03, 'alpha': 0.1, 'use_weights': True, 'check_estimator': True}
Melhor acurácia : 0.9380
Melhor precisão: 0.9535
Melhor recall  : 0.9123
Melhor F1      : 0.9324
Melhores parâmetros: {'n_cluster': 2, 'contamination': 0.04, 'alpha': 0.1, 'use_weights': True, 'check_estimator': True}
Melhor acurácia : 0.9413
Melhor precisão: 0.9761
Melhor recall  : 0.8967
Melhor F1      : 0.9347
Melhores parâmetros: {'n_cluste

KeyboardInterrupt: 

### Test

In [39]:
y_test_pred = model.predict(X_test)  # 0 = normal, 1 = anomaly

In [40]:
print_classification_report(y_test, y_test_pred)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.69      0.89      0.78     69245
         1.0       0.98      0.94      0.96    494928

    accuracy                           0.94    564173
   macro avg       0.84      0.92      0.87    564173
weighted avg       0.95      0.94      0.94    564173

Accuracy: 0.9376361506133757
Confusion Matrix:
 [[ 61690   7555]
 [ 27629 467299]]


## 3 - OCSVM

### Training

In [53]:
model = OneClassSVM(
    kernel='rbf', # More flexible non-linear kernel
    nu=0.01, # Maximum expected anomalies (~5%)
    gamma='scale', # Automatically adjust kernel width
    shrinking=True,
    cache_size=9999999, # In MB — optionally increase if you have memory
    max_iter=-1 # No limit on iterations
)

In [ ]:
model.fit(X_train)

### Validation

In [331]:
y_val_pred = model.predict(X_val)  # 0 = normal, 1 = anomaly
y_val_pred = np.where(y_val_pred == -1, 1, 0)

In [ ]:
print_classification_report(y_val, y_val_pred)

### Test

In [333]:
y_test_pred = model.predict(X_test)  # 0 = normal, 1 = anomaly
y_test_pred = np.where(y_test_pred == -1, 1, 0)
#y_test_score = model.decision_function(X_test)

In [ ]:
print_classification_report(y_test, y_test_pred)

## 4 - Isolation Forest

### Training

In [41]:
model = IForest(
    n_estimators=900,
    behaviour='new',
    max_samples='auto',
    contamination=0.05,
    max_features=1.0,
    bootstrap=False,
    random_state=RANDOM_SEED,
    verbose=1
)

In [42]:
model.fit(X_train)

IForest(behaviour='new', bootstrap=False, contamination=0.05,
    max_features=1.0, max_samples='auto', n_estimators=900, n_jobs=1,
    random_state=42, verbose=1)

### Validation

In [43]:
y_val_pred = model.predict(X_val)  # 0 = normal, 1 = anomaly

In [44]:
print_classification_report(y_val, y_val_pred)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.55      0.95      0.70     62321
         1.0       0.69      0.13      0.21     54992

    accuracy                           0.56    117313
   macro avg       0.62      0.54      0.46    117313
weighted avg       0.62      0.56      0.47    117313

Accuracy: 0.5640551345545677
Confusion Matrix:
 [[59216  3105]
 [48037  6955]]


### Test

In [45]:
y_test_pred = model.predict(X_test)  # 0 = normal, 1 = anomaly

In [46]:
print_classification_report(y_test, y_test_pred)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.13      0.95      0.23     69245
         1.0       0.95      0.13      0.22    494928

    accuracy                           0.23    564173
   macro avg       0.54      0.54      0.23    564173
weighted avg       0.85      0.23      0.22    564173

Accuracy: 0.22678681893674457
Confusion Matrix:
 [[ 65795   3450]
 [432776  62152]]


## 5 - HBOS

### Training

In [47]:
model = HBOS(n_bins=10, contamination=0.05)

In [48]:
model.fit(X_train)

HBOS(alpha=0.1, contamination=0.05, n_bins=10, tol=0.5)

### Validation

In [49]:
y_val_pred = model.predict(X_val)  # 0 = normal, 1 = anomaly

In [50]:
print_classification_report(y_val, y_val_pred)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.52      0.95      0.67     62321
         1.0       0.11      0.01      0.01     54992

    accuracy                           0.51    117313
   macro avg       0.32      0.48      0.34    117313
weighted avg       0.33      0.51      0.36    117313

Accuracy: 0.5079658690852676
Confusion Matrix:
 [[59193  3128]
 [54594   398]]


### Test

In [51]:
y_test_pred = model.predict(X_test)  # 0 = normal, 1 = anomaly

In [52]:
print_classification_report(y_test, y_test_pred)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.12      0.95      0.21     69245
         1.0       0.51      0.01      0.01    494928

    accuracy                           0.12    564173
   macro avg       0.32      0.48      0.11    564173
weighted avg       0.47      0.12      0.04    564173

Accuracy: 0.12307395072078954
Confusion Matrix:
 [[ 65832   3413]
 [491325   3603]]
